# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/WEEK-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

FlyRank editors can only review a small slice of a client's content library each week. Right now
there is no ranked queue telling them which pages to look at first — just the raw metrics. This
lane's question is: *which pages should be reviewed first for refresh, expansion, protection,
pruning, or monitoring?*

I picked this lane over the others for three reasons:

- It matches a real, recurring workflow (a content team picking their next N pages to fix), not
  just an analysis exercise — Lane 1 (signal analysis) explains *why* things move, but doesn't
  hand anyone a queue to act on today.
- The starter repo already ships a full working pipeline for exactly this lane
  (`scripts/01`-`05`), so I can see, on this data, whether a learned ranking actually beats a
  transparent rule before committing seven weeks to it (numbers below).
- It gives me a natural upgrade path for later weeks: the starter's label
  (`is_declining_label = trend_direction == "down"`) is a same-window proxy, not a future
  outcome. Once I move to the warehouse release, I can redefine the target as a genuine
  prior-90-days -> next-30-days label, which the lane guide flags as the stronger version of
  this exact lane.

This is a provisional choice — I can confirm or swap lanes through Week 4.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 10)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")


starter dataset: 30,000 rows x 44 columns, 32 clients


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** out of a client's whole content library, which pages should a
content editor put in front of them *this week* — and with what suggested action (refresh,
expand, protect, prune, or just monitor)?

**Who acts, and what do they do:** a FlyRank content strategist / editor works from a ranked
review queue. For a page near the top, they open it, read the reason codes attached to it
(e.g. "declining with demand", "thin but visible", "low CTR at a good position"), and decide
whether to rewrite/expand the content, fix the title or meta description, or leave it alone.
The output is a shortlist someone actually opens, not a raw table of 30,000 rows.

**Cost of a wrong call:**
- *False positive* (queue says "review this" but the page is actually fine): wastes a limited,
  expensive resource — editor hours — on a page that didn't need attention, and pushes a page
  that *did* need attention further down the list.
- *False negative* (a genuinely declining or high-opportunity page never surfaces): the page
  keeps losing visibility silently until someone notices by accident, which is the exact failure
  mode this queue exists to prevent.

Because editor time is the scarce resource, precision near the top of the queue (precision@20,
precision@50) matters more here than overall accuracy — a queue that's right about its top 50
picks is far more useful than one that's right "on average" across 30,000 pages nobody will look
at in order.

**Why data/ML helps, not just a simple rule:** a plain if-statement rule ("flag anything with
`days_since_last_update >= 180`") is a fine *baseline*, and I'll build one first. But the signal
that actually predicts trouble is spread across many correlated fields at once — impressions,
position, CTR, freshness, word count, engagement, all interacting differently by content type and
client — which is exactly the situation where a model beats a hand-written rule, *if* it can be
shown to beat that rule honestly (baseline-vs-model comparison numbers in the next section).

In [2]:
# supporting check for section 2: how big is the "worth reviewing" candidate pool,
# i.e. does the editor actually face a needle-in-a-haystack problem?
visible = df[df["impressions_90d"] >= 100]
print(f"pages with real search demand (impressions_90d >= 100): {len(visible):,} of {len(df):,} "
      f"({100*len(visible)/len(df):.1f}%)")
print("-> too many candidates for a human to triage one by one without a ranked queue.")


pages with real search demand (impressions_90d >= 100): 22,006 of 30,000 (73.4%)
-> too many candidates for a human to triage one by one without a ranked queue.


## 3. Quick look at the data (2-3 real numbers)

I ran the starter pipeline (`scripts/01`-`05`) earlier and I'm reusing its verified, committed
output (`outputs/model_report.md`) alongside a couple of fresh checks on the raw CSV below,
so the numbers here are consistent with each other.

In [3]:
# Number 1: how many candidate pages actually match simple, explainable trouble signals
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)            # avg_position == 0 means "no data", not rank zero -- excluded
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)                   # ctr is a x100 percentage: 0.5 means 0.5%, not 50%
)

print(f"declining_with_demand: {declining_with_demand.sum():,} rows "
      f"({100*declining_with_demand.mean():.1f}% of all pages)")
print(f"low_ctr_visible_page:  {low_ctr_visible.sum():,} rows "
      f"({100*low_ctr_visible.mean():.1f}% of all pages)")

# Number 2: does a learned ranking actually beat the plain rule on this data?
# (from outputs/model_report.md, generated by the starter pipeline on this same CSV)
print()
print("From the starter pipeline's verified model comparison (client-holdout validation):")
print("  baseline_rules   precision@50 = 0.240  (~12 of the top 50 review picks are right)")
print("  random_forest    precision@50 = 0.740  (~37 of the top 50 review picks are right)")


declining_with_demand: 13,152 rows (43.8% of all pages)
low_ctr_visible_page:  9,759 rows (32.5% of all pages)

From the starter pipeline's verified model comparison (client-holdout validation):
  baseline_rules   precision@50 = 0.240  (~12 of the top 50 review picks are right)
  random_forest    precision@50 = 0.740  (~37 of the top 50 review picks are right)


## 4. Careful words: what I can and can't claim

**What this work can say:**
- *Observed / decision-support*: which pages, based on measured signals from the trailing 90
  days, look most worth a human editor's next look — a ranked shortlist, not a verdict.
- *Directional*: that a learned ranking outperforms a hand-written rule on precision@K, on this
  data, under client-holdout validation (already true on the starter slice: 0.740 vs 0.240).
- The reason codes behind each recommendation, so a reviewer can check *why* a page was
  flagged rather than trusting a black-box number.

**What this work cannot say, and won't claim:**
- That refreshing a flagged page *will* cause a recovery — that needs a controlled experiment
  or a causal design this dataset alone can't provide, per the lane guide.
- Anything about Google's actual ranking algorithm or AI ranking/citation behavior — only
  observed clicks, impressions, and position are in this data.
- That the starter's `is_declining_label` (`trend_direction == "down"`, computed from the
  *current* window) is a real outcome measure. It's a same-window proxy label, and I'll say so
  explicitly wherever I use it, until I move to a genuine prior-window -> future-window label on
  the warehouse release.
- Anything about a specific client, page, or query — all IDs here are pseudonyms, joins/grouping
  only, never features, and never something I'll try to de-anonymize.

In [4]:
# nothing further to compute here -- this section is a written commitment, not a calculation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.